# 1. Prerequisites

In [ ]:
from utils.data import load_data, save_data
from utils.plotting import create_post_stim_raster_plot, map_colour_to_electrode

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import h5py
from tqdm import tqdm
from torch.utils.data import DataLoader, TensorDataset
import getpass

In [ ]:
# Load the specific network here
Network = 3
DIV     = 28
group_data  = False
test_data = False

In [ ]:
data = load_data(Network, DIV, group_data)
stimulation_parameters, stimulation_patterns, binned_spike_train_responses, stimulation_times, impedance_map, electrodes = data
print(f"Loaded data with response shape {binned_spike_train_responses.shape} and parameter shape {stimulation_parameters.shape}.")
if test_data:
    data = load_data(Network, DIV, group_data, test_mode=True)
    stimulation_parameters_test, stimulation_patterns_test, binned_spike_train_responses_test, stimulation_times_test, _, _ = data
else:
    binned_spike_train_responses_test = stimulation_patterns_test = stimulation_parameters_test = stimulation_times_test = None

# 2. Model Definitions and Data Preparation

In [ ]:
class CNN(nn.Module):
    """
    A simple CNN that takes an inputs of size (Batch, Channel, Height, Width). 
    The height in our task corresponds to the number of channels, the width to the number of time bins.
    """

    def __init__(self, n_outputs=1, n_base_channels= 32):
        super().__init__()

        self.convolutional_layers = nn.Sequential(
            nn.Conv2d(1, n_base_channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(n_base_channels, n_base_channels*2, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(n_base_channels*2, n_base_channels*4, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        # This function pools an arbitrary input size to a predefined size
        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))

        # output layers
        self.fully_connected_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n_base_channels * 4 * 4 * 4, n_base_channels*8),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(n_base_channels*8, n_outputs),
        )

    def forward(self, x):
        x = self.convolutional_layers(x)
        x = self.adaptive_pool(x) 
        x = self.fully_connected_layers(x)
        return x

In [ ]:
# Hyperparameters
# This is how often the model sees the full training dataset.  
n_epochs = 200

# The number of samples used for a single update step. Usually it is common to put this to large values from 32-1024. 
# It should be compared with the number of total training samples, and larger values will use more GPU memory. 
batch_size = 256

# Update step size which will be combined with the gradients. Small will have slow convergence, large will not converge. 
learning_rate = 1e-4 

# Here we define the used loss function. In this case it is cross entropy. The reduction we apply within the loop.
criterion = nn.CrossEntropyLoss(reduction='none')

# At last, we intialize the model and an optimizer, where latter takes care of combining learning rate and gradients.
n_classes = 16 # This is given by the data
# A CNN expects an input of shape (Batch, Channel, Height, Width). We have to manually add the channel dimension to the data:
if binned_spike_train_responses.ndim != 4:
    print(f"Initial input data size is {binned_spike_train_responses.shape}")
    binned_spike_train_responses = binned_spike_train_responses[:,None]
    print(f"Adjusted input data size is {binned_spike_train_responses.shape}")
if binned_spike_train_responses_test.ndim != 4: 
    binned_spike_train_responses_test = binned_spike_train_responses_test[:,None]
model = CNN(n_classes, n_base_channels=32)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
print(f"The model has {sum(p.numel() for p in model.parameters())} parameters.")

In [ ]:
# Data loading
# Pytorch dataloaders are parallel processing classes optimized for machine learning with Python. 
# It is recommended to use them. Here we set up such a dataloader. 
# We first split the training data into train and validation set. 
stimulation_patterns_one_hot = F.one_hot(torch.as_tensor(stimulation_patterns).long(), num_classes=n_classes).float()
val_frac = 0.1
n_val = int(binned_spike_train_responses.shape[0] * val_frac)
idx = torch.randperm(binned_spike_train_responses.shape[0])
val_idx = idx[:n_val]
trn_idx = idx[n_val:]
# Here, Z is any additional information we want to keep with the dataset split. 
Xtr, Ytr, Ztr = binned_spike_train_responses[trn_idx], stimulation_patterns_one_hot[trn_idx], stimulation_parameters[trn_idx]
Xval, Yval, Zval = binned_spike_train_responses[val_idx], stimulation_patterns_one_hot[val_idx], stimulation_parameters[val_idx]

# Here we initialize the dataloaders. 
train_dataset = TensorDataset(torch.as_tensor(Xtr, dtype=torch.float32), Ytr, torch.as_tensor(Ztr, dtype=torch.float32))
# Through deletion we can free up memory from our system
del Xtr, Ytr, Ztr
val_dataset   = TensorDataset(torch.as_tensor(Xval, dtype=torch.float32), Yval, torch.as_tensor(Zval, dtype=torch.float32))
del Xval, Yval, binned_spike_train_responses, stimulation_parameters, stimulation_patterns
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
if binned_spike_train_responses_test is not None:
    test_dataset   = TensorDataset(torch.as_tensor(binned_spike_train_responses_test, dtype=torch.float32))
    # Don't shuffle this data as else the predictions will be random w.r.t. the ground truth!
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    del binned_spike_train_responses_test, stimulation_parameters_test, stimulation_patterns_test
else:
    test_loader = None

# 3. Training and Validation Loop

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running with {device}.")
model.to(device)
best_val_loss = float("inf")
pbar = tqdm(range(0, n_epochs), leave=False)
train_losses = np.zeros(n_epochs)
validation_losses = dict()
validation_predictions = dict()

for epoch in pbar:
    model.train()
    train_loss = 0.0
    for x, y, z in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        pred = model(x)
        loss = torch.mean(criterion(pred, y))
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        
    train_loss /= len(train_loader)
    train_losses[epoch] = train_loss
    model.eval()
    val_loss = []
    val_pred = []
    
    # In a validation dataset, weights are not updated, hence we tell torch to not track any gradients for efficiency.
    with torch.no_grad():
        for x, y, z in val_loader:
            x, y = x.to(device), y.to(device)
            pred = model(x)
            loss = criterion(pred, y)
            val_loss.append(loss.numpy(force=True))
            # Here we will append the most likely class label through argmax
            val_pred.append(torch.argmax(pred, dim=-1).byte().numpy(force=True))
    val_loss = np.concatenate(val_loss)
    val_pred = np.concatenate(val_pred)
    avg_val_loss = np.mean(val_loss)
    validation_losses[f"epoch_{epoch+1}"] = val_loss
    validation_predictions[f"epoch_{epoch+1}"] = val_pred
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        # Here we save the currently best model. 
        # It's recommended to adapt this to use dedicated folders to track different tests. 
        torch.save(model.state_dict(), f"/home/{getpass.getuser()}/best_model.pth")
    pbar.set_postfix(loss_train=f"{train_loss:.4f}", loss_val=f"{avg_val_loss:.4f}", best_loss_val=f"{best_val_loss:.4f}")

# 4. Benchmarking

In [ ]:
# Per epoch average loss
val_epochs = epoch
epoch_losses = [np.mean(validation_losses[f"epoch_{e+1}"]) for e in range(val_epochs)]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, val_epochs + 1), epoch_losses, marker='o', markersize=3)
ax.set_xlabel("Epoch")
ax.set_ylabel("Mean loss")
ax.set_title("Validation loss per epoch")
fig.tight_layout()
plt.show()

# Last epoch: calculate accuracy in %
last_losses = np.array(validation_predictions[f"epoch_{val_epochs}"]) == Zval[:,1]
frequencies = Zval[:, 0]

# per frequency loss
unique_freqs = np.unique(frequencies)
pattern_losses = np.array([last_losses[frequencies == f].mean() * 100 for f in unique_freqs])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(unique_freqs, pattern_losses, marker='o', markersize=3, label="Pattern Loss")
ax.set_xlabel("Frequency [Hz]")
ax.set_ylabel("Accuracy [%]")
ax.set_title(f"Loss per frequency")
fig.tight_layout()
plt.show()

# mask to only look at samples up to max_freq
max_freq = 6
patterns = Zval[:, 1]
mask = frequencies <= max_freq
losses_masked = last_losses[mask]
patterns_masked = patterns[mask]

# per pattern loss
unique_patterns = np.unique(patterns_masked)
pattern_losses = np.array([losses_masked[patterns_masked == p].mean() * 100 for p in unique_patterns])

x = np.arange(len(unique_patterns))
width = 0.35
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x, pattern_losses, width, label="Pattern Loss")
ax.set_xticks(x)
ax.set_xticklabels(unique_patterns.astype(int).astype(str))
ax.set_xlabel("Pattern")
ax.set_ylabel("Accuracy [%]")
ax.set_title(f"Loss per pattern (freq ≤ {max_freq})")
fig.tight_layout()
plt.show()

# 5. Submitting predictions

In [ ]:
# If we have an additional test dataset, where we do not have ground truth, we run and save it here. 
# First we load the best model according to the validation loss. 
model.load_state_dict(torch.load(f"/home/{getpass.getuser()}/best_model.pth", weights_only=True, map_location=device))
if test_loader is not None:
    predictions = []
    with torch.no_grad():
        for x, in test_loader:
            x = x.to(device)
            pred = model(x).argmax(dim=-1).numpy(force=True)
            predictions.append(pred)
    predictions = np.concatenate(predictions, axis=0)
    save_data(
        network = Network, 
        day_in_vitro = DIV, 
        binned_spike_train_responses = None, 
        stimulation_parameters = None, 
        stimulation_patterns = predictions,
        use_group_data = group_data,
        test_mode = test_data
    )